# Notebook 04b — CMF Post-Hoc Weight Tuning

**Paper:** *An Illusion of Unlearning? Assessing Machine Unlearning Through Internal Representations*  
(Gao, Unal, Rangamani, Zhu — AISTATS 2026 · arXiv:2604.08271v1)

**Dependency & Protocol:**
- **Depends directly on `04_cmf_static.ipynb` checkpoints.** This notebook loads the Stage-1 unlearned weights produced by `04_cmf_static.ipynb` rather than retraining Stage 1 from scratch.
- For each loaded checkpoint: freezes the entire encoder permanently, promotes the CMF weight to a trainable parameter using `CMFWeightsTrainable`.
- Runs $k_{posthoc}$ epochs ($k=2$ and $k=5$) of gradient descent on $W$ using cross-entropy on `retain_only` and `retain_plus_forget` data.
- **Sanity Check:** Explicitly compares Probe and NCC accuracy before vs. after Stage 2. Since the encoder remains strictly frozen, Probe and NCC representations must remain IDENTICAL, while Output-level accuracy adapts.
- Saves checkpoints as `{base_method}_cmf_posthoc_k{k}_{phase2_data}_{mean_source}_class{forget_class}_seed{seed}.pt`.

In [ ]:
import subprocess, sys
def sh(cmd, verbose=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if verbose and r.stdout: print(r.stdout[-4000:])
    if r.returncode != 0 and r.stderr: print('STDERR:', r.stderr[-2000:])
    return r.returncode
sh('pip install -q timm einops scikit-learn matplotlib seaborn pytorch-lightning torchmetrics')

In [ ]:
import os, sys, json, random, math, time, copy, traceback
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib; import matplotlib.pyplot as plt
matplotlib.rcParams.update({'figure.dpi': 110})
print('PyTorch:', torch.__version__, '  CUDA:', torch.cuda.is_available())

In [ ]:
REPO_DIR = '/kaggle/working/CMF_Unlearning'
if not os.path.isdir(REPO_DIR):
    sh(f'git clone https://github.com/tiensinh2/CMF_Unlearning.git {REPO_DIR}')
else:
    sh(f'git -C {REPO_DIR} remote set-url origin https://github.com/tiensinh2/CMF_Unlearning.git')
    sh(f'git -C {REPO_DIR} pull origin main')
os.chdir(REPO_DIR); sys.path.insert(0, REPO_DIR)
result = subprocess.run(['git', '-C', REPO_DIR, 'rev-parse', 'HEAD'],
                        capture_output=True, text=True)
REPO_COMMIT = result.stdout.strip() or 'main'
print('Repo commit:', REPO_COMMIT)

In [ ]:
CKPT_DATASET_DIR = '/kaggle/input/datasets/btk23021592/cmf-notebook1'
_CONFIG_CANDIDATES = [
    f'{CKPT_DATASET_DIR}/cmf_benchmark_config.json',
    f'{CKPT_DATASET_DIR}/checkpoints/cmf_benchmark/cmf_benchmark_config.json',
    f'{CKPT_DATASET_DIR}/cmf_benchmark/cmf_benchmark_config.json',
    './notebooks/Result_nb1/checkpoints/cmf_benchmark/cmf_benchmark_config.json',
]
config_path = CKPT_ROOT_NB1 = None
for _p in _CONFIG_CANDIDATES:
    if os.path.exists(_p):
        config_path = _p; CKPT_ROOT_NB1 = os.path.dirname(_p); break

if config_path and os.path.exists(config_path):
    with open(config_path) as f: NB1_CFG = json.load(f)
    DATASET     = NB1_CFG.get('dataset', 'cifar10')
    ARCH        = NB1_CFG.get('arch', 'resnet18')
    NUM_CLASSES = NB1_CFG.get('num_classes', 10)
    TEST_MODE   = NB1_CFG.get('test_mode', False)
else:
    DATASET     = 'cifar100'
    ARCH        = 'resnet18'
    NUM_CLASSES = 100
    TEST_MODE   = False
    CKPT_ROOT_NB1 = '/kaggle/working/checkpoints/cmf_benchmark'

STAGE = '4b'
# Paper §A.3 / Table 1&3: CIFAR-100 uses 5 single-class forget sets
# config.py UNLEARN_SCENARIOS['cifar100']['single'] = [[0],[1],[2],[3],[5]]
FORGET_CLASSES = [0, 1, 2, 3, 5]   # paper CIFAR-100 single-class scenario
SEEDS          = [0]
BASE_METHODS   = ['grad_ascent_descent', 'random_label', 'salun', 'scrub', 'tarun']
MEAN_SOURCES   = ['train']
K_VALUES       = [2, 5]
PHASE2_DATA_OPTIONS = ['retain_only', 'retain_plus_forget']
POSTHOC_LR     = 1e-3

# Path to NB4a output dataset (attach it in Kaggle UI as input).
# Update this to match the exact mount path of your attached dataset.
CKPT_ROOT_NB4A = '/kaggle/input/datasets/nguyenhunguet/cmf-static-notebook4/checkpoints/cmf_paper'
CKPT_ROOT      = '/kaggle/working/checkpoints/cmf_posthoc_paper'
os.makedirs(f'{CKPT_ROOT}/cmf_posthoc', exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'STAGE={STAGE}  DATASET={DATASET}  ARCH={ARCH}  device={device}')
print(f'CKPT_ROOT_NB4A exists: {os.path.isdir(CKPT_ROOT_NB4A)}')
print(f'K_VALUES={K_VALUES}  PHASE2_DATA={PHASE2_DATA_OPTIONS}')

In [ ]:
import torchvision, torchvision.transforms as transforms
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4), transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

full_train      = torchvision.datasets.CIFAR100('/kaggle/working/data', train=True,
                                                download=True,  transform=transform_train)
full_train_eval = torchvision.datasets.CIFAR100('/kaggle/working/data', train=True,
                                                download=False, transform=transform_test)
test_set        = torchvision.datasets.CIFAR100('/kaggle/working/data', train=False,
                                                download=True, transform=transform_test)

# Pre-index by class label
test_targets = torch.tensor(test_set.targets)
TEST_CLASS_IDX = {
    c: (test_targets == c).nonzero(as_tuple=True)[0].tolist()
    for c in range(NUM_CLASSES)
}
train_targets = torch.tensor(full_train.targets)
TRAIN_CLASS_IDX = {
    c: (train_targets == c).nonzero(as_tuple=True)[0].tolist()
    for c in range(NUM_CLASSES)
}
print(f'Train size: {len(full_train)}  Test size: {len(test_set)}')

In [ ]:
import argparse
from unlearn.cmf_weights import ModelModule
from unlearn.cmf_two_stage import CMFWeightsTrainable

def build_cmf_model(args):
    """Construct proper ModelModule with CMF classifier head."""
    model = ModelModule(args).to(device)
    assert hasattr(model, 'CMFweights'), 'Model must have CMFweights attribute!'
    return model

def make_cmf_args(base_method, lr, epochs, mean_source, forget_class,
                  forget_train_idx, retain_train_idx, seed=0):
    return argparse.Namespace(
        dataset=DATASET, arch=ARCH, num_classes=NUM_CLASSES,
        class_label_names=list(range(NUM_CLASSES)),
        unlearn_method=f'{base_method}_CMF_RemoveFC',
        unlearn_class=[forget_class],
        batch_size=128, test_batch_size=256, lr=lr,
        momentum=0.9, weight_decay=5e-4, epochs_or_steps=epochs,
        seed=seed,
        num_retain_samples=len(retain_train_idx),
        num_forget_samples=len(forget_train_idx),
        grad_norm_clip=1.0,
        SVD_alpha_r=1000, SVD_alpha_f=30,
        SVD_samples=900, SVD_max_patches=10000,
        freeze_except_last=False,
        scrub_del_bsz=64, scrub_sgda_bsz=64, scrub_msteps=2, scrub_epochs=epochs,
        salun_threshold=0.5,
        tarun_impair_lr=lr, tarun_samples_per_class=1000,
        dry_run=TEST_MODE, no_cuda=False, no_mps=True, gamma=0.5,
        data_path='/kaggle/working/data', remove_FC=True,
        CMFClassifier=True, CMF_momentum=0.9, pretrained=False, temperature=1.0,
        prob_batch_size=256, lp_every=0, mean_source=mean_source,
        repo_commit=REPO_COMMIT, test_mode=TEST_MODE,
    )

@torch.no_grad()
def eval_acc(model, loader):
    model.eval()
    correct = total = 0
    for x, y in loader:
        correct += (model(x.to(device)).argmax(1).cpu() == y).sum().item()
        total   += y.size(0)
    return 100.0 * correct / max(total, 1)

@torch.no_grad()
def cmf_extract_features(model, loader):
    model.eval()
    feats, labs = [], []
    for x, y in loader:
        x = x.to(device)
        f = model.extract_features(x)
        z = model.extract_features(x)
        feats.append(z.cpu()); labs.append(y)
    return torch.cat(feats), torch.cat(labs)

def run_probe_cmf(model, train_retain_ldr, train_forget_ldr,
                  test_retain_ldr, test_forget_ldr, n_epochs=None, seed=42):
    if n_epochs is None:
        n_epochs = 200 if DATASET.lower() == 'cifar100' else 50
    torch.manual_seed(seed); np.random.seed(seed)
    Xtr, ytr = cmf_extract_features(model, train_retain_ldr)
    Xfg, yfg = cmf_extract_features(model, train_forget_ldr)
    Xall = torch.cat([Xtr, Xfg])
    yall = torch.cat([ytr, yfg])
    head = nn.Linear(Xall.size(1), NUM_CLASSES).to(device)
    opt  = optim.SGD(head.parameters(), lr=1e-2, momentum=0.9)
    ldr  = torch.utils.data.DataLoader(
               torch.utils.data.TensorDataset(Xall, yall), batch_size=256, shuffle=True)
    for _ in range(n_epochs):
        head.train()
        for bx, by in ldr:
            opt.zero_grad()
            F.cross_entropy(head(bx.to(device)), by.to(device)).backward()
            opt.step()
    head.eval()
    with torch.no_grad():
        Xte_r, yte_r = cmf_extract_features(model, test_retain_ldr)
        Xte_f, yte_f = cmf_extract_features(model, test_forget_ldr)
        ret_acc = (head(Xte_r.to(device)).argmax(1).cpu() == yte_r).float().mean().item() * 100
        fgt_acc = (head(Xte_f.to(device)).argmax(1).cpu() == yte_f).float().mean().item() * 100
    return ret_acc, fgt_acc

def run_ncc_cmf(model, train_retain_ldr, train_forget_ldr,
                test_retain_ldr, test_forget_ldr):
    Xtr, ytr = cmf_extract_features(model, train_retain_ldr)
    Xfg, yfg = cmf_extract_features(model, train_forget_ldr)
    Xall = torch.cat([Xtr, Xfg])
    yall = torch.cat([ytr, yfg])
    means = []
    for c in range(NUM_CLASSES):
        mask = (yall == c)
        mu = Xall[mask].mean(0) if mask.any() else torch.zeros(Xall.size(1))
        means.append(mu)
    M = torch.stack(means)
    Xte_r, yte_r = cmf_extract_features(model, test_retain_ldr)
    Xte_f, yte_f = cmf_extract_features(model, test_forget_ldr)
    ret_pred = torch.cdist(Xte_r.unsqueeze(0), M.unsqueeze(0)).squeeze(0).argmin(1)
    fgt_pred = torch.cdist(Xte_f.unsqueeze(0), M.unsqueeze(0)).squeeze(0).argmin(1)
    return ((ret_pred == yte_r).float().mean().item() * 100,
            (fgt_pred == yte_f).float().mean().item() * 100)

def eval_cmf_three_metrics(model,
                           test_retain_ldr, test_forget_ldr,
                           train_retain_eval_ldr, train_forget_eval_ldr):
    out_ret = eval_acc(model, test_retain_ldr)
    out_fgt = eval_acc(model, test_forget_ldr)
    lp_ret, lp_fgt   = run_probe_cmf(model, train_retain_eval_ldr, train_forget_eval_ldr,
                                      test_retain_ldr, test_forget_ldr)
    ncc_ret, ncc_fgt = run_ncc_cmf(model, train_retain_eval_ldr, train_forget_eval_ldr,
                                    test_retain_ldr, test_forget_ldr)
    return {'output_retain_acc': out_ret, 'output_forget_acc': out_fgt,
            'probe_retain_acc':  lp_ret,  'probe_forget_acc':  lp_fgt,
            'ncc_retain_acc':    ncc_ret, 'ncc_forget_acc':    ncc_fgt}

print('Helpers ready.')

In [ ]:
import os
print(f'Contents of CKPT_ROOT_NB4A ({CKPT_ROOT_NB4A}):')
for root, dirs, files in os.walk(CKPT_ROOT_NB4A):
    level = root.replace(CKPT_ROOT_NB4A, '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    if level < 2:
        for f in files[:10]:
            print(f'{indent}  {f}')
        if len(files) > 10:
            print(f'{indent}  ... ({len(files)} files total)')


In [ ]:
results_4b = []

for forget_class in FORGET_CLASSES:
    for seed in SEEDS:
        forget_train_idx = TRAIN_CLASS_IDX[forget_class]
        retain_train_idx = [
            i for c in range(NUM_CLASSES)
            if c != forget_class
            for i in TRAIN_CLASS_IDX[c]
        ]
        retain_loader = torch.utils.data.DataLoader(
            torch.utils.data.Subset(full_train, retain_train_idx),
            batch_size=128, shuffle=True, num_workers=2)
        forget_loader = torch.utils.data.DataLoader(
            torch.utils.data.Subset(full_train, forget_train_idx),
            batch_size=128, shuffle=True, num_workers=2)
        retain_forget_loader = torch.utils.data.DataLoader(
            torch.utils.data.Subset(full_train, retain_train_idx + forget_train_idx),
            batch_size=128, shuffle=True, num_workers=2)
        full_train_loader = torch.utils.data.DataLoader(
            full_train, batch_size=256, shuffle=False, num_workers=2)
        train_retain_eval_ldr = torch.utils.data.DataLoader(
            torch.utils.data.Subset(full_train_eval, retain_train_idx),
            batch_size=256, shuffle=False, num_workers=2)
        train_forget_eval_ldr = torch.utils.data.DataLoader(
            torch.utils.data.Subset(full_train_eval, forget_train_idx),
            batch_size=256, shuffle=False, num_workers=2)
        test_forget_idx = TEST_CLASS_IDX[forget_class]
        test_retain_idx = [
            i for c in range(NUM_CLASSES)
            if c != forget_class
            for i in TEST_CLASS_IDX[c]
        ]
        test_forget_ldr = torch.utils.data.DataLoader(
            torch.utils.data.Subset(test_set, test_forget_idx),
            batch_size=256, shuffle=False, num_workers=2)
        test_retain_ldr = torch.utils.data.DataLoader(
            torch.utils.data.Subset(test_set, test_retain_idx),
            batch_size=256, shuffle=False, num_workers=2)
        test_loader_full = torch.utils.data.DataLoader(
            test_set, batch_size=256, shuffle=False, num_workers=2)

        for base_method in BASE_METHODS:
            for mean_source in MEAN_SOURCES:
                # ── 1. Find and load the Stage-1 cmf_static checkpoint ──────────────
                s1_tag = f'{base_method}_cmf_static_{mean_source}_class{forget_class}_seed{seed}'
                if TEST_MODE: s1_tag += '_testmode'
                s1_candidates = [
                    f'{CKPT_ROOT_NB4A}/cmf_static/{s1_tag}.pt',
                    f'{CKPT_ROOT_NB4A}/{s1_tag}.pt',
                    f'./notebooks/Result_nb4_static/checkpoints/{s1_tag}.pt',
                    f'./checkpoints/cmf_paper/cmf_static/{s1_tag}.pt',
                ]
                s1_path = next((p for p in s1_candidates if os.path.exists(p)), None)
                if s1_path is None:
                    print(f'Stage 1 checkpoint not found for {s1_tag}. Skipping.')
                    continue

                print(f'\n[Loaded S1] {s1_path}')
                ck_s1 = torch.load(s1_path, map_location=device)
                s1_state = ck_s1.get('model_state_dict', ck_s1)

                for k in K_VALUES:
                    for phase2_data in PHASE2_DATA_OPTIONS:
                        tag = f'{base_method}_cmf_posthoc_k{k}_{phase2_data}_{mean_source}_class{forget_class}_seed{seed}'
                        if TEST_MODE: tag += '_testmode'
                        ckpt_path = f'{CKPT_ROOT}/cmf_posthoc/{tag}.pt'

                        if os.path.exists(ckpt_path):
                            print(f'[{tag}] exists — loading cached result.')
                            ck = torch.load(ckpt_path, map_location=device)
                            results_4b.append(ck['metrics'])
                            continue

                        print(f'\n[{tag}] Running Stage 2 (k={k}, data={phase2_data})...')
                        torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)

                        args = make_cmf_args(base_method, POSTHOC_LR, k, mean_source,
                                             forget_class, forget_train_idx, retain_train_idx,
                                             seed=seed)
                        model = build_cmf_model(args)
                        model.load_state_dict(s1_state, strict=False)

                        # ── Baseline metrics before Stage 2 ─────────────────────────
                        metrics_before = eval_cmf_three_metrics(
                            model,
                            test_retain_ldr, test_forget_ldr,
                            train_retain_eval_ldr, train_forget_eval_ldr
                        )

                        # ── Freeze entire encoder permanently ──────────────────────
                        for p in model.parameters():
                            p.requires_grad_(False)

                        # ── Promote CMF weights cleanly via CMFWeightsTrainable ─────
                        tw = CMFWeightsTrainable(model.CMFweights)
                        tw.promote()
                        optim_w = torch.optim.SGD([tw.W_param], lr=POSTHOC_LR, momentum=0.9, weight_decay=1e-4)

                        s2_loader = retain_forget_loader if phase2_data == 'retain_plus_forget' else retain_loader

                        t0 = time.time()
                        try:
                            # Encoder stays in eval mode throughout Stage 2.
                            # model.train() would put BatchNorm into training mode,
                            # updating running_mean/var and changing extract_features output.
                            model.eval()
                            for ep in range(1, k + 1):
                                for xb, yb in s2_loader:
                                    xb, yb = xb.to(device), yb.to(device)
                                    optim_w.zero_grad()
                                    with torch.no_grad():
                                        f = model.extract_features(xb)
                                        z = model.extract_features(xb)
                                    logits = tw(z, 1.0)
                                    loss = F.cross_entropy(logits, yb)
                                    loss.backward()
                                    optim_w.step()
                                    if TEST_MODE:
                                        break

                            wall_min = (time.time() - t0) / 60

                            # Write final W_param back into CMFweights.weight
                            # so model.forward() uses the trained weights.
                            tw.sync_back()

                            # ── Metrics after Stage 2 ──────────────────────────────────
                            metrics = eval_cmf_three_metrics(
                                model,
                                test_retain_ldr, test_forget_ldr,
                                train_retain_eval_ldr, train_forget_eval_ldr
                            )

                            # ── Sanity Check: Probe & NCC must be invariant ────────────
                            probe_diff = abs(metrics['probe_forget_acc'] - metrics_before['probe_forget_acc'])
                            ncc_diff   = abs(metrics['ncc_forget_acc'] - metrics_before['ncc_forget_acc'])
                            print(f'  [Sanity Check] Probe forget diff: {probe_diff:.4f}%, NCC forget diff: {ncc_diff:.4f}%')
                            if probe_diff > 1e-3 or ncc_diff > 1e-3:
                                print('  WARNING: Encoder representation changed during Stage 2! (Check freeze logic)')
                                metrics['invariance_violated'] = True
                                raise RuntimeError(
                                    f'Invariance check failed: probe_diff={probe_diff:.4f}%, ncc_diff={ncc_diff:.4f}% (threshold 1e-3)'
                                )
                            else:
                                metrics['invariance_violated'] = False

                            metrics.update({
                                'method': base_method, 'mean_source': mean_source,
                                'forget_class': forget_class, 'seed': seed, 'stage': STAGE,
                                'k_posthoc': k, 'phase2_data': phase2_data, 'lr': POSTHOC_LR,
                                'wall_clock_minutes': wall_min,
                                'protocol': 'whole_class_single',
                            })

                            torch.save({
                                'model_state_dict': model.state_dict(),
                                'config': {
                                    'base_method': base_method, 'mean_source': mean_source,
                                    'dataset': DATASET, 'arch': ARCH, 'num_classes': NUM_CLASSES,
                                    'forget_class': forget_class, 'seed': seed,
                                    'k_posthoc': k, 'phase2_data': phase2_data, 'lr': POSTHOC_LR,
                                    'protocol': 'whole_class_single',
                                    'repo_commit': REPO_COMMIT, 'test_mode': TEST_MODE,
                                },
                                'seed': seed, 'metrics': metrics,
                            }, ckpt_path)
                            print(f'  Saved {ckpt_path}')
                            print(f'  out   R={metrics["output_retain_acc"]:.2f}%  F={metrics["output_forget_acc"]:.2f}%')
                            print(f'  probe R={metrics["probe_retain_acc"]:.2f}%  F={metrics["probe_forget_acc"]:.2f}%')
                            print(f'  ncc   R={metrics["ncc_retain_acc"]:.2f}%  F={metrics["ncc_forget_acc"]:.2f}%')
                            results_4b.append(metrics)
                        except Exception as e:
                            print(f'ERROR during Stage 2: {e}')
                            traceback.print_exc()
                            continue

df_4b = pd.DataFrame(results_4b)
csv_path = f'{CKPT_ROOT}/results_4b_cmf_posthoc.csv'
df_4b.to_csv(csv_path, index=False)
print(f'\nResults saved to {csv_path}')
if not df_4b.empty:
    metric_cols = ['output_retain_acc','output_forget_acc',
                   'probe_retain_acc','probe_forget_acc','ncc_retain_acc','ncc_forget_acc']
    print('\n=== Summary ===')
    print(df_4b.groupby(['method', 'k_posthoc', 'phase2_data'])[metric_cols].mean().round(2).to_string())